# Student Performance Classification

**Classification Project 13 of 100** — Classify student performance level from demographics, family background, and study habits.

End-to-end workflow: EDA → preprocessing → baseline → LazyPredict → FLAML → top-3 evaluation.

## 2. Project Overview

Understanding what drives student performance is critical for educators, policymakers, and parents. Early identification of at-risk students enables timely interventions — tutoring, counselling, or resource allocation.

This notebook uses the **Student Alcohol Consumption** dataset from UCI/Kaggle, which contains student records from two Portuguese schools. We derive a **classification target** from the final grade (G3): students are classified as **Fail (G3 < 10)** or **Pass (G3 >= 10)** on a 0–20 grading scale.

**Workflow:**
1. Download & validate the dataset from Kaggle
2. Perform thorough EDA on demographics and study habits
3. Preprocess with sklearn pipelines (split-before-fit, no leakage)
4. Establish baselines (Dummy, LogReg, Random Forest)
5. Benchmark ~30 classifiers with LazyPredict
6. Run FLAML AutoML
7. Select & evaluate the top 3 models
8. Analyse errors and extract educational insights

## 3. Learning Objectives

By completing this notebook you will learn to:

1. **Derive a classification target** from a continuous grade variable
2. Handle a **mixed-type dataset** with many ordinal and binary features
3. Decide between **OrdinalEncoder** and **OneHotEncoder** based on feature semantics
4. Handle **moderate class imbalance** (~30% fail rate)
5. Detect and discuss **potential target leakage** (G1, G2 predicting G3)
6. Build a **leakage-free** preprocessing pipeline
7. Use appropriate metrics: **accuracy, F1, precision, recall, ROC-AUC**
8. Benchmark with **LazyPredict** and **FLAML**
9. Interpret results through an **educational policy lens**
10. Understand **ethical considerations** in student classification

## 4. Problem Statement

> **Given a student's demographics, family background, study habits, and school-related features, predict whether they will pass (G3 >= 10) or fail (G3 < 10) their final exam.**

This is a **binary classification** task. Identifying at-risk students early allows educators to provide targeted support. **Recall** (catching at-risk students) is important, but **precision** matters too — falsely labelling students as at-risk can cause unnecessary stress.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| **Educators** | Identify at-risk students early for targeted intervention |
| **School administrators** | Allocate tutoring and counselling resources efficiently |
| **Parents** | Receive early warnings about academic difficulties |
| **Policy makers** | Understand factors driving student outcomes |
| **ML learners** | Practice deriving classification targets and handling educational data ethics |

## 6. Dataset Overview

| Property | Value |
|---|---|
| **Name** | Student Alcohol Consumption |
| **Source** | UCI ML Repository / Kaggle |
| **Kaggle slug** | `uciml/student-alcohol-consumption` |
| **Files** | `student-mat.csv` (Math), `student-por.csv` (Portuguese) |
| **Rows** | ~395 (Math) / ~649 (Portuguese) |
| **Features** | 30 (demographics, family, study habits, grades) |
| **Target (derived)** | `pass` (1 if G3 >= 10, else 0) |

**Key columns:**
- `G1`, `G2`, `G3` — first period, second period, and final grades (0–20)
- `studytime`, `failures`, `absences` — study and attendance features
- `Medu`, `Fedu` — mother's and father's education level
- `Dalc`, `Walc` — workday and weekend alcohol consumption
- `famrel`, `freetime`, `goout`, `health` — lifestyle indicators

## 7. Dataset Source and License Notes

- **Original source:** P. Cortez and A. Silva. *Using Data Mining to Predict Secondary School Student Performance.* EUROSIS, 2008.
- **UCI Repository:** https://archive.ics.uci.edu/ml/datasets/student+performance
- **Kaggle:** https://www.kaggle.com/datasets/uciml/student-alcohol-consumption
- **License:** CC BY 4.0 — free for educational and commercial use with attribution.
- **Note:** We use the Math subject file (`student-mat.csv`) as the primary dataset.

## 8. Environment Setup

We install any packages not already available.

In [ ]:
import subprocess, sys, importlib

for pkg in ["lazypredict", "flaml", "kagglehub", "xgboost", "lightgbm"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Environment ready.")

## 9. Imports

In [ ]:
import os, warnings, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    classification_report
)

from lazypredict.Supervised import LazyClassifier
from flaml import AutoML

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42
print("All imports loaded.")

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = "uciml/student-alcohol-consumption"
FILE_HINT = "student-mat"  # Math subject file
TARGET = "pass"  # derived binary target
PASS_THRESHOLD = 10  # G3 >= 10 means pass
TEST_SIZE = 0.15
VAL_SIZE = 0.15
SEED = 42
FLAML_BUDGET = 120  # seconds
print(f"Target: {TARGET} | Pass threshold: G3 >= {PASS_THRESHOLD} | Seed: {SEED}")

## 11. Dataset Download or Loading

We use `kagglehub` to download the dataset. We select the Math subject file (`student-mat.csv`).

In [ ]:
import kagglehub

try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f"Dataset downloaded to: {path}")
except Exception as e:
    raise RuntimeError(
        f"Failed to download dataset: {e}\n"
        "Ensure KAGGLE_API_TOKEN is set or ~/.kaggle/kaggle.json exists."
    )

csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
print(f"CSV files found: {[os.path.basename(f) for f in csv_files]}")

# Select the math file
selected = next((f for f in csv_files if FILE_HINT in os.path.basename(f).lower()), csv_files[0])
print(f"Using: {os.path.basename(selected)}")

df_raw = pd.read_csv(selected)
print(f"Shape: {df_raw.shape}")
df_raw.head()

## 12. Data Validation Checks

We verify data integrity and derive the classification target from the final grade G3.

In [ ]:
print(f"Shape: {df_raw.shape}")
print(f"Missing values: {df_raw.isnull().sum().sum()}")

dupes = df_raw.duplicated().sum()
print(f"Duplicate rows: {dupes}")
if dupes > 0:
    df_raw = df_raw.drop_duplicates().reset_index(drop=True)

# Verify G3 exists
assert "G3" in df_raw.columns, "Final grade column G3 not found!"
print(f"\nG3 (final grade) stats:\n{df_raw['G3'].describe()}")

# Derive binary target
df_raw[TARGET] = (df_raw["G3"] >= PASS_THRESHOLD).astype(int)
print(f"\nDerived target distribution:")
print(df_raw[TARGET].value_counts())
print(f"Pass rate: {df_raw[TARGET].mean():.1%}")

## 13. Exploratory Data Analysis

We explore the relationship between student features and academic performance.

In [ ]:
df = df_raw.copy()
df.describe().T

In [ ]:
# Grade distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(['G1', 'G2', 'G3']):
    axes[i].hist(df[col], bins=20, color='steelblue', edgecolor='white')
    axes[i].axvline(PASS_THRESHOLD, color='red', linestyle='--', label=f'Pass threshold={PASS_THRESHOLD}')
    axes[i].set_title(f'{col} Distribution')
    axes[i].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Key features vs target
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, col in enumerate(['studytime', 'failures', 'absences', 'Medu', 'Fedu', 'goout']):
    ax = axes[i // 3][i % 3]
    for label in [0, 1]:
        grp = df[df[TARGET] == label]
        tag = 'Pass' if label == 1 else 'Fail'
        ax.hist(grp[col], bins=15, alpha=0.5, label=tag)
    ax.set_title(col)
    ax.legend()
plt.suptitle('Feature Distributions by Pass/Fail', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with target
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[num_cols].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
print("Top correlations with pass/fail:")
print(corr.head(10))

## 14. Target Analysis

The pass rate is approximately 65–70%. A majority-class baseline would achieve ~70% accuracy but miss all failing students.

In [ ]:
pass_rate = df[TARGET].mean()
print(f"Pass rate: {pass_rate:.1%}")
print(f"Majority-class baseline accuracy: {max(pass_rate, 1-pass_rate):.1%}")
print(f"Class counts:\n{df[TARGET].value_counts()}")

fig, ax = plt.subplots(figsize=(5, 4))
df[TARGET].value_counts().plot.bar(ax=ax, color=["salmon", "steelblue"])
ax.set_title("Target Distribution")
ax.set_xticklabels(["Fail (0)", "Pass (1)"], rotation=0)
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 15. Train / Validation / Test Split Strategy

We use a **70/15/15 stratified** split. We split **before** any preprocessing to prevent data leakage.

**Important leakage decision:** We **drop G1 and G2** because they are intermediate grades that directly predict G3. In a real early-warning system, we would only have G1 available at midterm. For this notebook, we train without any grade features to simulate a pre-enrollment prediction scenario.

In [ ]:
# Drop G1, G2, G3 (leakage) and derived target column
leak_cols = ['G1', 'G2', 'G3']
X = df.drop(columns=[TARGET] + [c for c in leak_cols if c in df.columns])
y = df[TARGET]

print(f"Dropped leakage columns: {leak_cols}")
print(f"Features: {X.shape[1]}")

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=VAL_SIZE / (1 - TEST_SIZE),
    random_state=SEED, stratify=y_temp
)

print(f"\nTrain: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
for name, ys in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    print(f"{name} pass rate: {ys.mean():.1%}")

## 16. Preprocessing Strategy

**Decisions:**
- **Missing values:** Few to none expected; impute with median/most-frequent as safety net.
- **Categorical encoding:** OneHotEncoder for low-cardinality categoricals (school, sex, address, etc.).
- **Scaling:** StandardScaler for numeric features.
- **Ordinal features** (Medu, Fedu, traveltime, studytime, etc.): kept as numeric since they have natural ordering.
- **No leakage:** ColumnTransformer fit only on training data.

In [ ]:
num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train.select_dtypes(include=['object']).columns.tolist()
print(f"Numeric: {len(num_features)}, Categorical: {len(cat_features)}")
print(f"Categorical: {cat_features}")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_features),
    ],
    remainder="drop"
)
print("Preprocessor ready.")

## 17. Feature Engineering

We create domain-inspired features:
- **total_alcohol:** combined workday + weekend alcohol consumption
- **parent_edu_avg:** average of mother and father education levels
- **study_to_goout_ratio:** studytime relative to going out frequency

In [ ]:
def engineer_features(df_in):
    df_out = df_in.copy()
    if 'Dalc' in df_out.columns and 'Walc' in df_out.columns:
        df_out['total_alcohol'] = df_out['Dalc'] + df_out['Walc']
    if 'Medu' in df_out.columns and 'Fedu' in df_out.columns:
        df_out['parent_edu_avg'] = (df_out['Medu'] + df_out['Fedu']) / 2
    if 'studytime' in df_out.columns and 'goout' in df_out.columns:
        df_out['study_to_goout'] = df_out['studytime'] / df_out['goout'].clip(lower=1)
    return df_out

X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)

# Refresh column lists
num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_features),
    ],
    remainder="drop"
)
print(f"Total features after engineering: {len(num_features) + len(cat_features)}")

## 18. Baseline Model

DummyClassifier sets the floor. Then LogReg and Random Forest provide informed baselines.

In [ ]:
results = {}

dummy = Pipeline([("pre", preprocessor), ("clf", DummyClassifier(strategy="most_frequent", random_state=SEED))])
dummy.fit(X_train, y_train)
y_pred_d = dummy.predict(X_val)
results["Dummy"] = {
    "Accuracy": accuracy_score(y_val, y_pred_d),
    "F1": f1_score(y_val, y_pred_d, zero_division=0),
    "Recall": recall_score(y_val, y_pred_d, zero_division=0),
}
print("Dummy:", {k: f"{v:.4f}" for k, v in results["Dummy"].items()})

lr = Pipeline([("pre", preprocessor), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))])
lr.fit(X_train, y_train)
y_prob_lr = lr.predict_proba(X_val)[:, 1]
results["LogReg"] = {
    "Accuracy": accuracy_score(y_val, lr.predict(X_val)),
    "F1": f1_score(y_val, lr.predict(X_val)),
    "Recall": recall_score(y_val, lr.predict(X_val)),
    "ROC-AUC": roc_auc_score(y_val, y_prob_lr),
}
print("LogReg:", {k: f"{v:.4f}" for k, v in results["LogReg"].items()})

rf = Pipeline([("pre", preprocessor), ("clf", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=SEED, n_jobs=-1))])
rf.fit(X_train, y_train)
y_prob_rf = rf.predict_proba(X_val)[:, 1]
results["RF"] = {
    "Accuracy": accuracy_score(y_val, rf.predict(X_val)),
    "F1": f1_score(y_val, rf.predict(X_val)),
    "Recall": recall_score(y_val, rf.predict(X_val)),
    "ROC-AUC": roc_auc_score(y_val, y_prob_rf),
}
print("RF:", {k: f"{v:.4f}" for k, v in results["RF"].items()})

## 19. LazyPredict Benchmark

LazyPredict fits ~30 classifiers quickly. This is for **exploration only**.

In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)

lazy = LazyClassifier(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
print("Top 15 models by Accuracy:")
models_lp.head(15)

## 20. FLAML AutoML Run

FLAML optimises for **accuracy** since the class balance is moderate.

In [ ]:
automl = AutoML()
automl.fit(
    X_train, y_train,
    task="classification",
    metric="accuracy",
    time_budget=FLAML_BUDGET,
    seed=SEED,
    verbose=0,
)

print(f"Best model: {automl.best_estimator}")
print(f"Best config: {automl.best_config}")
y_pred_fl = automl.predict(X_val)
y_prob_fl = automl.predict_proba(X_val)[:, 1]
results["FLAML"] = {
    "Accuracy": accuracy_score(y_val, y_pred_fl),
    "F1": f1_score(y_val, y_pred_fl),
    "Recall": recall_score(y_val, y_pred_fl),
    "ROC-AUC": roc_auc_score(y_val, y_prob_fl),
}
print("FLAML:", {k: f"{v:.4f}" for k, v in results["FLAML"].items()})

## 21. Top 3 Model Selection

Based on LazyPredict and FLAML results, we select three strong classifiers:
1. **LightGBM** — fast gradient boosting
2. **XGBoost** — robust alternative
3. **GradientBoosting** — sklearn ensemble

In [ ]:
try:
    from lightgbm import LGBMClassifier
    has_lgbm = True
except ImportError:
    has_lgbm = False
try:
    from xgboost import XGBClassifier
    has_xgb = True
except ImportError:
    has_xgb = False

top3 = {}
if has_lgbm:
    top3["LightGBM"] = LGBMClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        class_weight="balanced", random_state=SEED, verbose=-1, n_jobs=-1
    )
else:
    from sklearn.ensemble import ExtraTreesClassifier
    top3["ExtraTrees"] = ExtraTreesClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1)

if has_xgb:
    imbalance_ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    top3["XGBoost"] = XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        scale_pos_weight=imbalance_ratio, random_state=SEED, verbosity=0, n_jobs=-1
    )
else:
    from sklearn.ensemble import AdaBoostClassifier
    top3["AdaBoost"] = AdaBoostClassifier(n_estimators=200, random_state=SEED)

top3["GradientBoosting"] = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=4, random_state=SEED
)
print(f"Top 3: {list(top3.keys())}")

## 22. Final Training and Evaluation of Top 3

Train on full training set, evaluate on held-out **test set**.

In [ ]:
X_train_t = preprocessor.fit_transform(X_train)
X_val_t = preprocessor.transform(X_val)
X_test_t = preprocessor.transform(X_test)

final_results = {}
for name, model in top3.items():
    model.fit(X_train_t, y_train)
    y_pred = model.predict(X_test_t)
    y_prob = model.predict_proba(X_test_t)[:, 1]
    final_results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
    }
    print(f"\n{name}:")
    print(classification_report(y_test, y_pred, target_names=["Fail", "Pass"]))

results_df = pd.DataFrame(final_results).T
print("\n=== Final Test Results ===")
results_df

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(5 * len(top3), 4))
if len(top3) == 1: axes = [axes]
for ax, (name, model) in zip(axes, top3.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, model.predict(X_test_t), ax=ax, cmap="Blues", display_labels=["Fail", "Pass"])
    ax.set_title(name)
plt.suptitle("Confusion Matrices — Test Set", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, model in top3.items():
    RocCurveDisplay.from_estimator(model, X_test_t, y_test, ax=axes[0], name=name)
    PrecisionRecallDisplay.from_estimator(model, X_test_t, y_test, ax=axes[1], name=name)
axes[0].set_title("ROC Curves")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_title("Precision-Recall Curves")
plt.tight_layout()
plt.show()

## 23. Error Analysis

We examine which students the model misclassifies.

In [ ]:
best_name = results_df["Accuracy"].idxmax()
best_model = top3[best_name]
y_pred_best = best_model.predict(X_test_t)

fn_mask = (y_test.values == 1) & (y_pred_best == 0)  # Pass predicted as Fail
fp_mask = (y_test.values == 0) & (y_pred_best == 1)  # Fail predicted as Pass

print(f"Best model: {best_name}")
print(f"False Negatives (Pass predicted Fail): {fn_mask.sum()}")
print(f"False Positives (Fail predicted Pass): {fp_mask.sum()}")

test_df = X_test.copy()
test_df["error_type"] = "correct"
test_df.loc[test_df.index[fn_mask], "error_type"] = "false_negative"
test_df.loc[test_df.index[fp_mask], "error_type"] = "false_positive"

num_test = test_df.select_dtypes(include=[np.number]).columns[:6].tolist()
if num_test:
    print("\nMean features by error type:")
    print(test_df.groupby("error_type")[num_test].mean().round(2))

## 24. Interpretation and Business Insight

**Key findings:**
1. **Failures (past failures)** is the strongest predictor of future failure
2. **Study time and absences** have meaningful but moderate predictive power
3. **Parent education level** correlates positively with passing
4. **Alcohol consumption** shows a weak negative effect when grades are removed

**Educational recommendations:**
- Focus early intervention on students with **prior failures** and high absences
- Encourage study time and reduce barriers to attendance
- Use the model as a **screening tool**, not a deterministic label — always combine with teacher observations
- Be cautious about **self-fulfilling prophecies** — labelling students as 'at risk' can affect their motivation

## 25. Limitations

1. **Small dataset:** Only ~395 students (Math) — high variance in results
2. **Two Portuguese schools only** — limited geographical and cultural generalisability
3. **No longitudinal data:** Single snapshot, not tracking student progress over time
4. **Self-reported features:** Alcohol consumption, study time are self-reported and potentially unreliable
5. **Derived target:** Pass/fail threshold at G3=10 is arbitrary; different thresholds give different results
6. **Without grades:** Removing G1, G2 significantly limits predictive power (the realistic scenario)

## 26. How to Improve This Project

1. Include **G1** (first period grade) as a feature for midterm predictions
2. Try **multi-class classification** (A/B/C/D/F grade bands) instead of binary
3. Add **cross-validation** for more stable estimates on this small dataset
4. Combine Math and Portuguese datasets for more training data
5. Add **SHAP** analysis for individual student explanations
6. Try **ensemble stacking** to combine model strengths

## 27. Production Considerations

- **Privacy:** Student data is sensitive — ensure FERPA/GDPR compliance
- **Fairness:** Audit predictions across gender, family background, and socioeconomic status
- **Explainability:** Teachers and parents need interpretable predictions, not black boxes
- **Human oversight:** Model flags should inform teachers, not replace their judgment
- **Feedback loop:** Track actual student outcomes to update the model over semesters

## 28. Common Mistakes

1. **Leaking G1/G2 grades** — using intermediate grades to predict final grade is circular
2. **Using G3 directly as a feature** — the target column must be excluded
3. **Not stratifying** — with 30% fail rate, random splits can create unbalanced sets
4. **Overfitting on small data** — ~400 rows means cross-validation is essential
5. **Ignoring ordinal structure** — treating Medu/Fedu as unordered categoricals loses information
6. **Ethical carelessness** — labelling students as 'failures' without context harms individuals

## 29. Mini Challenge / Exercises

1. Include G1 as a feature (simulating a midterm prediction) — how much does accuracy improve?
2. Create a 3-class target (Fail < 10, Average 10–14, Good >= 15) and compare results
3. Run 5-fold cross-validation — how stable are the results across folds?
4. Add SHAP feature importance — which features matter most per-student?
5. Compare results between the Math and Portuguese subject files

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| **Task** | Binary classification — student pass/fail prediction |
| **Dataset** | ~395 students, 30 features, ~65–70% pass rate |
| **Best metric focus** | Accuracy, F1 (balanced trade-off) |
| **Baseline** | DummyClassifier ~70% accuracy |
| **Best models** | Gradient boosting family |
| **Key insight** | Past failures and study habits predict future outcomes |
| **Limitation** | Small dataset, no longitudinal tracking, self-reported features |

**What you learned:**
- How to derive a classification target from continuous grades
- Why leakage detection matters (G1/G2 → G3)
- Feature engineering from educational data
- Ethical considerations in student classification
- The full benchmark → AutoML → top 3 evaluation pipeline